[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# Selecting Rows &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with the catalog's models and the twelve books loaded. Run
it first. Every task below only reads, so they can be run in any order.


In [1]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (CharField, ForeignKeyField, IntegerField, Model, OperationalError,
                    SqliteDatabase, fn)

AUTHORS = [                                                         # name, the year of the first book
    ("Ursula Vance", 2014),
    ("Marco Pietra", 2009),
    ("Ines O'Brien", 1998),
    ("Kofi Mensah", 2015),
]

BOOKS = [                                                           # title, author, year, pages
    ("The Salt Road", "Ursula Vance", 2014, 312),
    ("Nightjar", "Ursula Vance", 2018, 244),
    ("The Quiet Engine", "Ursula Vance", 2021, 398),
    ("Stone and Tide", "Marco Pietra", 2009, 501),
    ("The Lantern Keeper", "Marco Pietra", 2016, 276),
    ("Riverwork", "Marco Pietra", 2022, 189),
    ("A Careful Fire", "Ines O'Brien", 1998, 420),
    ("The Long Field", "Ines O'Brien", 2004, 355),
    ("Winter Harbour", "Ines O'Brien", 2011, 263),
    ("The Drum Line", "Kofi Mensah", 2015, 198),
    ("Harmattan", "Kofi Mensah", 2019, 331),
    ("Small Machines", "Kofi Mensah", 2023, 287),
]

def sql(query):
    """The SQL a query will send, and the values that go with it, on one line."""
    statement, values = query.sql()
    return " ".join(statement.split()) + (f"  {values}" if values else "")

db = SqliteDatabase(":memory:")


class CatalogModel(Model):
    """Every model in the catalog names the database once, here."""

    class Meta:
        database = db


class Author(CatalogModel):
    name = CharField(max_length=60, unique=True)
    first_book = IntegerField()


class Book(CatalogModel):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()

def build(database):
    """Create the tables and load the catalog, in one transaction."""
    database.create_tables([Author, Book])
    with database.atomic():
        Author.insert_many([{"name": name, "first_book": year} for name, year in AUTHORS]).execute()
        written = {author.name: author.id for author in Author.select()}
        Book.insert_many([{"title": title, "author": written[author], "year": year, "pages": pages}
                          for title, author, year, pages in BOOKS]).execute()


build(db)
print("peewee", peewee.__version__, "| the catalog:", Author.select().count(), "authors and",
      Book.select().count(), "books")


peewee 4.5.1 | the catalog: 4 authors and 12 books


**1.** The long recent books, SQL first.


In [2]:
wanted = Book.select().where((Book.year >= 2015) & (Book.pages > 250))

print(sql(wanted))
print([(book.title, book.year, book.pages) for book in wanted])


SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages" FROM "book" AS "t1" WHERE (("t1"."year" >= ?) AND ("t1"."pages" > ?))  [2015, 250]
[('The Lantern Keeper', 2016, 276), ('Harmattan', 2019, 331), ('The Quiet Engine', 2021, 398), ('Small Machines', 2023, 287)]


Two conditions, two pairs of parentheses, and both of them in the `WHERE` clause where they belong.


**2.** The same filter with `and`, and what it lost.


In [3]:
lost = Book.select().where(Book.year >= 2015 and Book.pages > 250)

print(sql(lost))
print("rows:", len(lost), "against", len(wanted), "| oldest returned:",
      min(book.year for book in lost))


SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages" FROM "book" AS "t1" WHERE ("t1"."pages" > ?)  [250]
rows: 9 against 4 | oldest returned: 1998


The year condition is the one that was lost: `and` returned its right operand, so only
`pages > 250` reached peewee. The proof is in the `WHERE` clause, which names one column, and in the
oldest year that came back, which is well before 2015.


**3.** Each author's earliest book.


In [4]:
earliest = (Author.select(Author.name, fn.MIN(Book.year).alias("first"))
                  .join(Book)
                  .group_by(Author.name)
                  .order_by(fn.MIN(Book.year)))

for row in earliest:
    print(f"  {row.name:<15} {row.first}")


  Ines O'Brien    1998
  Marco Pietra    2009
  Ursula Vance    2014
  Kofi Mensah     2015


`fn.MIN` is computed per group, and ordering by the same expression sorts the groups by it. The
`alias` is what makes `row.first` readable.


**4.** The authors with more than two books.


In [5]:
busy = (Author.select(Author.name, fn.COUNT(Book.id).alias("books"))
              .join(Book)
              .where(Book.year >= 2010)                             # which rows are counted
              .group_by(Author.name)
              .having(fn.COUNT(Book.id) > 2)                        # which groups are kept
              .order_by(Author.name))

print(sql(busy))
print([(row.name, row.books) for row in busy])


SELECT "t1"."name", COUNT("t2"."id") AS "books" FROM "author" AS "t1" INNER JOIN "book" AS "t2" ON ("t2"."author_id" = "t1"."id") WHERE ("t2"."year" >= ?) GROUP BY "t1"."name" HAVING (COUNT("t2"."id") > ?) ORDER BY "t1"."name"  [2010, 2]
[('Kofi Mensah', 3), ('Ursula Vance', 3)]


Every author has three books in all, so without the `where` this would return all four. The `where`
cuts the rows down to those from 2010 on before they are grouped, and the `having` then reads a
count of what survived. The condition on a count can only be a `having`, because a `where` runs
before there is a count to test.


**5.** A book that is not there, asked for safely.


In [6]:
title = "The Book That Was Never Written"

print("get_or_none:", Book.get_or_none(Book.title == title))
print("first()    :", Book.select().where(Book.title == title).first())
print("count()    :", Book.select().where(Book.title == title).count())


get_or_none: None
first()    : None
count()    : 0


Both give `None` rather than raising. `count()` is the third way to ask, and returns `0`, which is
the right call when the row itself is not wanted.


**6.** The whole catalog in pages of five, by the last `id` seen.


In [7]:
seen, pages, after = [], [], None
while True:
    rows = list(Book.select().where(Book.id > after).order_by(Book.id).limit(5)
                if after is not None else Book.select().order_by(Book.id).limit(5))
    if not rows:
        break
    after = rows[-1].id
    pages.append([book.title for book in rows])
    seen.extend(book.title for book in rows)

for number, page in enumerate(pages, start=1):
    print(f"  page {number}: {page}")
print("titles seen:", len(seen), "| distinct:", len(set(seen)), "| books:", Book.select().count())


  page 1: ['The Salt Road', 'Nightjar', 'The Quiet Engine', 'Stone and Tide', 'The Lantern Keeper']
  page 2: ['Riverwork', 'A Careful Fire', 'The Long Field', 'Winter Harbour', 'The Drum Line']
  page 3: ['Harmattan', 'Small Machines']
titles seen: 12 | distinct: 12 | books: 12


The walk stops when a page comes back empty, and the counts agree: every book was seen once. The
`id` is a good key to page by because it is unique, so there is never a tie to break.


---

&#8592; **Back to:** [Selecting Rows](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/04-selecting-rows.ipynb)  &nbsp;&middot;&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
